# Stock factor evaluation

Evaluate whether a per-stock factor (one value per `(date, asset_id)`)
carries cross-sectional return predictability.


## Factor type

This recipe uses
`AnalysisConfig.individual_continuous(metric=Metric.IC)` —
axes `(FactorScope.INDIVIDUAL, Signal.CONTINUOUS, Metric.IC)`.

Procedure: per-date Spearman ρ between factor and forward return,
aggregated to a Newey-West HAC t-statistic on `E[IC]`. PANEL mode
(`N ≥ 2`); `N = 1` raises `ModeAxisError` since there is no
cross-section to rank within.

Literature: [Grinold (1989)](../reference/bibliography.md);
[Newey & West (1987)](../reference/bibliography.md).


## Use this when

- Factor varies across assets at each date (per-stock signal, e.g.
  momentum, value, quality).
- Cross-section is wide (`N ≥ 30` for clean inference; `10 ≤ N < 30`
  emits `BORDERLINE_CROSS_SECTION_N`).
- Time series is at least 30 periods (`T < 20` is hard-blocked).

## What it tests

Null hypothesis `E[IC] = 0` — the factor has no rank-based
predictive ordering of forward returns across assets, on average
across dates. Standard error is NW HAC over the per-date IC series.

## Output to read

1. `profile.verdict()` — `pass` / `fail` against `primary_p < 0.05`.
2. `profile.primary_p` — directly the IC NW HAC p-value.
3. `profile.stats[StatCode.IC_MEAN]` — sign + magnitude of average IC.
4. `profile.stats[StatCode.IC_T_NW]` — the t-statistic that produced
   the p-value. Compare to ±2 as a rough sanity check.
5. `profile.warnings` — `UNRELIABLE_SE_SHORT_PERIODS` etc. flag risks
   that do *not* enter `verdict()` but should change interpretation.


## 1. Setup


In [1]:
from __future__ import annotations

import factrix as fl
from factrix.preprocess.returns import compute_forward_return

print('factrix version:', fl.__version__)


factrix version: 0.7.0


## 2. Synthesise a cross-sectional panel

`make_cs_panel` produces a canonical panel with a target IC built
in. `ic_target=0.08` is a realistic effect size for a working
single-factor strategy.


In [2]:
raw = fl.datasets.make_cs_panel(
    n_assets=100,
    n_dates=500,
    ic_target=0.08,
    seed=2024,
)
panel = compute_forward_return(raw, forward_periods=5)
print(f"panel shape={panel.shape}  N={panel['asset_id'].n_unique()}")


panel shape=(49400, 5)  N=100


## 3. Evaluate

One factory call, one `evaluate()`. The factory commits to the
three axes; `evaluate()` derives `Mode` from the panel shape and
dispatches to the registered procedure.


In [3]:
cfg = fl.AnalysisConfig.individual_continuous(
    metric=fl.Metric.IC,
    forward_periods=5,
)
profile = fl.evaluate(panel, cfg)

print(f"verdict      = {profile.verdict()}")
print(f"primary_p    = {profile.primary_p:.4g}")
print(f"mode         = {profile.mode}")
print(f"ic_mean      = {profile.stats[fl.StatCode.IC_MEAN]:+.4f}")
print(f"ic_t_nw      = {profile.stats[fl.StatCode.IC_T_NW]:+.2f}")
print(f"nw_lags_used = {profile.stats[fl.StatCode.NW_LAGS_USED]:.0f}")


verdict      = pass
primary_p    = 2.129e-40
mode         = panel
ic_mean      = +0.0722
ic_t_nw      = +14.60
nw_lags_used = 5


## 4. Inspect the full diagnose dict

`diagnose()` is the structured triage interface — same content the
verdict and individual-stat reads above derive from, in one dict
for human inspection or AI agent consumption.


In [4]:
import json

print(json.dumps(profile.diagnose(), indent=2, default=str))


{
  "mode": "panel",
  "n_obs": 494,
  "n_assets": 100,
  "primary_p": 2.1286625029236503e-40,
  "warnings": [],
  "info_notes": [],
  "stats": {
    "ic_mean": 0.0722106866557101,
    "ic_t_nw": 14.6039191698416,
    "ic_p": 2.1286625029236503e-40,
    "nw_lags_used": 5.0
  }
}


## 5. Sample-guard edge cases

This recipe runs the happy path. For the full `n_assets` × factory
behaviour matrix (small-N warnings, `N=1` fallbacks, `T<20` hard
block) see
[Guides § PANEL vs TIMESERIES](../guides/panel-timeseries.md).
Two notes for this cell specifically:

- `N < 30` emits `BORDERLINE_CROSS_SECTION_N` / `SMALL_CROSS_SECTION_N`.
- `N = 1` raises `ModeAxisError` with `suggested_fix=common_continuous(...)`
  — the factor would no longer be cross-sectional.


In [5]:
print('stock_factor_evaluation: ok')


stock_factor_evaluation: ok
